In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, accuracy_score, classification_report ,recall_score,precision_score , f1_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
df=pd.read_csv("../data/processed-data/cleaned_data.csv")

In [34]:
X = df.drop(columns=["Churn"],
            errors="ignore")
y=df["Churn"]


In [35]:
df["Churn"].value_counts()*100/df.shape[0]

Churn
0    73.421502
1    26.578498
Name: count, dtype: float64

In [36]:
x1=df.memory_usage(deep=True).sum() / 1024 ** 2

In [37]:

df = df.apply(
    lambda col: (
        col.astype("category")
        if col.dtype.name in ["object", "str", "string"]
        else col.astype("int8")
        if col.name != "TotalCharges"
        else col
    )
)

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   gender            7032 non-null   category
 1   SeniorCitizen     7032 non-null   int8    
 2   Partner           7032 non-null   int8    
 3   Dependents        7032 non-null   int8    
 4   tenure            7032 non-null   int8    
 5   PhoneService      7032 non-null   int8    
 6   MultipleLines     7032 non-null   category
 7   InternetService   7032 non-null   category
 8   OnlineSecurity    7032 non-null   category
 9   OnlineBackup      7032 non-null   category
 10  DeviceProtection  7032 non-null   category
 11  TechSupport       7032 non-null   category
 12  StreamingTV       7032 non-null   category
 13  StreamingMovies   7032 non-null   category
 14  Contract          7032 non-null   category
 15  PaperlessBilling  7032 non-null   int8    
 16  PaymentMethod     7032 non-null   c

In [39]:
y1=df.memory_usage(deep=True).sum() / 1024 ** 2
print(f"end={x1} MB, optimized={y1} MB, diff={x1-y1} MB")

end=4.641109466552734 MB, optimized=0.1831226348876953 MB, diff=4.457986831665039 MB


In [40]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=2)

In [41]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5625, 19)
(1407, 19)
(5625,)
(1407,)


In [42]:
print(X_train[["TotalCharges", "MonthlyCharges", "tenure"]].dtypes)

TotalCharges      float64
MonthlyCharges    float64
tenure              int64
dtype: object


In [43]:
cat_cols = X_train.select_dtypes(
    include=["object", "category", "bool","str"]
).columns.tolist()
num_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()
cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore",drop="first"))
])
num_pipeline = Pipeline([
    ("scaler", StandardScaler())
])
preprocessor = ColumnTransformer([
    ("cat", cat_pipeline, cat_cols),
    ("num", num_pipeline, num_cols)
])

In [44]:
model_ = Pipeline([
    ("preprocessor", preprocessor),
    ("lr", LogisticRegression(max_iter=1000))
])
import time
t_start=time.time()
model_.fit(X_train, y_train)
y_pred_= model_.predict(X_test)
print(time.time()-t_start)
lr1 = model_.named_steps["lr"]

0.09066247940063477


In [45]:
print("Train accuracy:", model_.score(X_train, y_train))
print("Test accuracy:", model_.score(X_test, y_test))
print("Precision:", precision_score(y_test, y_pred_, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred_, zero_division=0))
print("Number of coefficients:", lr1.coef_.size)

Train accuracy: 0.8053333333333333
Test accuracy: 0.8052594171997157
Precision: 0.6497005988023952
Recall: 0.5802139037433155
F1 Score: 0.6129943502824858
Number of coefficients: 30


# use polynomial feature without PCA

In [46]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lr", LogisticRegression(C=4.0, max_iter=1000))
])
t_start=time.time()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(time.time()-t_start)
lr = model.named_steps["lr"]


1.1993982791900635


In [47]:
#with out pca 
print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))
print("Number of coefficients:", lr.coef_.size)

Train accuracy: 0.8167111111111112
Test accuracy: 0.7960199004975125
Precision: 0.638095238095238
Recall: 0.5374331550802139
F1 Score: 0.5834542815674891
Number of coefficients: 495


# using pca for matrix reduction and for better performance


In [51]:
pca_model = Pipeline([
    ("preprocessor", preprocessor),
    ("poly",PolynomialFeatures(degree=3,include_bias=False)),
    ("pca", PCA(n_components=0.95)),
    ("lr1", LogisticRegression(C=6.0,max_iter=1000))
])
t_start=time.time()
pca_model.fit(X_train, y_train)
y_pred_custom = pca_model.predict(X_test)
print(time.time()-t_start)
lr1 = pca_model.named_steps["lr1"]

118.4957754611969


In [49]:
# Evaluate using pca
print("Train accuracy:", pca_model.score(X_train, y_train))
print("Accuracy:", accuracy_score(y_test, y_pred_custom))
print("Precision:", precision_score(y_test, y_pred_custom))
print("Recall:", recall_score(y_test, y_pred_custom))
print("F1 score:", f1_score(y_test, y_pred_custom))
print("Number of coefficients:", lr1.coef_.size)

Train accuracy: 0.8110222222222222
Accuracy: 0.8052594171997157
Precision: 0.6655629139072847
Recall: 0.5374331550802139
F1 score: 0.5946745562130178
Number of coefficients: 90


## classification report

In [50]:
print(classification_report(y_test, y_pred))
print("after pca model:")
print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.64      0.54      0.58       374

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.79      0.80      0.79      1407

after pca model:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1033
           1       0.67      0.54      0.59       374

    accuracy                           0.81      1407
   macro avg       0.75      0.72      0.73      1407
weighted avg       0.80      0.81      0.80      1407



## Conclusion
Both models have the same test accuracy of 80.53%.
- Polynomial Logistic Regression with PCA: Has slightly higher precision (66.56%), meaning its positive churn predictions are more often correct. However, its recall (53.74%) and F1 score (59.47%) are lower.
- Normal Logistic Regression: Has higher recall (58.02%) and F1 score (61.30%), so it identifies more customers who will churn and provides a better balance between precision and recall.
- Model complexity: The polynomial model has 90 coefficients, compared with 30 for normal logistic regression, making it more complex.
Final conclusion: Normal Logistic Regression is the better choice for this project because it achieves the same test accuracy, a higher F1 score, and better recall with fewer coefficients. Since identifying customers likely to leave is important in churn prediction, its higher recall is especially useful.